Before you can run this code, your labels must be uploaded as an asset in GEE.

In [ ]:
import ee

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='superfund-featurizing') ## project path in GEE
import numpy as np


THIS CELL FOR WHEN THERE IS NO DATE COLUMN:


In [ ]:

ee.Initialize(project='superfund-featurizing')
import numpy as np


import os
from matplotlib import pyplot as plt

import pickle
import pandas as pd
import numpy as np
import shapely
import geopandas as gpd
import ee # earth engine
import folium

from shapely.geometry import Point



DEBUG = False


## must change to .01/.001 depending
grid_delta = .01

bands = np.arange(0,64).astype(str)
bands = ["A0" + x if len(x)==1 else "A"+x for x in bands]



# Combined reducer. Processes each band that we want to summarize over the given rectangle
reducer = (ee.Reducer.mean()
           .combine(ee.Reducer.percentile([0,10,20,30,40,50,60,70,80,90,100]), '', True))



sparse_coords_ee_asset_name = "projects/superfund-featurizing/assets/federal_superfund_labels_test"
points = ee.FeatureCollection(sparse_coords_ee_asset_name)
points = points.toList(points.size())

if DEBUG:
    points = points.slice(0,1_000)

def make_rect(feature):
    lon = ee.Number(feature.get('lon'))
    lat = ee.Number(feature.get('lat'))

    # Using 0.005 creates a total span of 0.01 degrees (~1.1km), so for  .001 degrees must use .0005
    rect = ee.Geometry.Rectangle([
        lon.subtract(0.05), lat.subtract(0.05),
        lon.add(0.05), lat.add(0.05)
    ])

    return ee.Feature(rect, feature.toDictionary())




batch_size = 500  # features per batch
total = 7777
if DEBUG:
    total = 1_000
num_batches = (total // batch_size) + 1

print(f"Total features: {total}, batches: {num_batches}")

# Loop over batches and export each one
for i in range(num_batches):
    start = i * batch_size
    end = start + batch_size
    subset = points.slice(start,end)
    subset = ee.FeatureCollection(subset)

    fname = f'federal_superfund_test_01_grid{i}'


    rects = subset.map(make_rect)

    collection = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
              .filterDate("2025-01-01")
              .filterBounds(rects)
             )

    composite = collection.select(bands).median() ## Should just be one image anyway. Median over time dimension

    # Reduce per-rect ## make sure you change scale depending on if you're making .01 or .001
    stats_per_rect = composite.reduceRegions(
        collection=rects,
        reducer=reducer,
        scale=1000,
        crs='EPSG:4326',
    )

    ## Start download

    task = ee.batch.Export.table.toDrive(
        collection=stats_per_rect,
        description=f'federal_superfund_test_featurized{i}',
        fileNamePrefix=fname,
        fileFormat='CSV')

    task.start()
    print('Export task started:', task.status())



Total features: 7777, batches: 16
Export task started: {'state': 'READY', 'description': 'federal_superfund_test_featurized0', 'priority': 100, 'creation_timestamp_ms': 1780522438047, 'update_timestamp_ms': 1780522438047, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'VBEVF2S5GC6TE2X7RI3W64GW', 'name': 'projects/superfund-featurizing/operations/VBEVF2S5GC6TE2X7RI3W64GW'}
Export task started: {'state': 'READY', 'description': 'federal_superfund_test_featurized1', 'priority': 100, 'creation_timestamp_ms': 1780522438846, 'update_timestamp_ms': 1780522438846, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'KAE2GUP6AOZO7445KHRIHPZO', 'name': 'projects/superfund-featurizing/operations/KAE2GUP6AOZO7445KHRIHPZO'}
Export task started: {'state': 'READY', 'description': 'federal_superfund_test_featurized2', 'priority': 100, 'creation_timestamp_ms': 1780522439547, 'update_timestamp_ms': 1780522439547, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': '

USE THIS CODE IF THERE IS A YEAR COLUMN:

In [ ]:
import os
from matplotlib import pyplot as plt

import pickle
import pandas as pd
import numpy as np
import shapely
import geopandas as gpd
import ee # earth engine
import folium


from shapely.geometry import Point



DEBUG = False


## must change to .01/.001 depending
grid_delta = .01

bands = np.arange(0,64).astype(str)
bands = ["A0" + x if len(x)==1 else "A"+x for x in bands]



ee.Initialize(project='subtle-bit-479914-n9')


# Combined reducer. Processes each band that we want to summarize over the given rectangle
reducer = (ee.Reducer.mean()
           .combine(ee.Reducer.percentile([0,10,20,30,40,50,60,70,80,90,100]), '', True))


sparse_coords_ee_asset_name = "projects/subtle-bit-479914-n9/assets/SF_thru_2017_01_200p_with_date_labels_for_GEE" ##change this line to be the name of your asset
points = ee.FeatureCollection(sparse_coords_ee_asset_name)
points = points.toList(points.size())

if DEBUG:
    points = points.slice(0,1_000)

def make_rect(feature):
    lon = ee.Number(feature.get('lon'))
    lat = ee.Number(feature.get('lat'))

    # Using 0.005 creates a total span of 0.01 degrees (~1.1km), so for  .001 degrees must use .0005
    rect = ee.Geometry.Rectangle([
        lon.subtract(0.005), lat.subtract(0.005),
        lon.add(0.005), lat.add(0.005)
    ])

    return ee.Feature(rect, feature.toDictionary())


# In[20]:


batch_size = 500  # features per batch
total = 1935 ## look in GEE and see how many rows there are for a given asset
if DEBUG:
    total = 1_000
num_batches = (total // batch_size) + 1

print(f"Total features: {total}, batches: {num_batches}")

# Loop over batches and export each one
for i in range(num_batches):
    start = i * batch_size
    end = start + batch_size
    subset = points.slice(start,end)
    subset = ee.FeatureCollection(subset)

    fname = f'SF_2017_200p_01_grid_date_labeled{i}' ## update output path name


    rects = subset.map(make_rect)

    collection = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
              .filterDate("2017-01-01") ##change to be the year you want the imagery to be from
              .filterBounds(rects)
             )

    composite = collection.select(bands).median() ## Should just be one image anyway. Median over time dimension

    # Reduce per-rect ## make sure you change scale depending on if you're making .01 or .001
    stats_per_rect = composite.reduceRegions(
        collection=rects,
        reducer=reducer,
        scale=1000,
        crs='EPSG:4326',
    )
    def preserve_metadata(f) if there's a discovery year
        return f.set('DISCOVERY_YEAR', f.get('DISCOVERY_YEAR'))

    stats_with_metadata = stats_per_rect.map(preserve_metadata)
     # Start download

    task = ee.batch.Export.table.toDrive(
        collection=stats_with_metadata,
        description=f'SF_01_200percent_2017_date_label_grid{i}',
        fileNamePrefix=fname,
        fileFormat='CSV')

    task.start()
    print('Export task started:', task.status())



Total features: 1935, batches: 4
Export task started: {'state': 'READY', 'description': 'SF_01_200percent_2026_date_label_grid0', 'priority': 100, 'creation_timestamp_ms': 1775756442147, 'update_timestamp_ms': 1775756442147, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'VZUEXG23WIL77WSZPFJOYFDL', 'name': 'projects/subtle-bit-479914-n9/operations/VZUEXG23WIL77WSZPFJOYFDL'}
Export task started: {'state': 'READY', 'description': 'SF_01_200percent_2026_date_label_grid1', 'priority': 100, 'creation_timestamp_ms': 1775756442787, 'update_timestamp_ms': 1775756442787, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'ILM75DBHOVRUIG5XHKFRUP2O', 'name': 'projects/subtle-bit-479914-n9/operations/ILM75DBHOVRUIG5XHKFRUP2O'}
Export task started: {'state': 'READY', 'description': 'SF_01_200percent_2026_date_label_grid2', 'priority': 100, 'creation_timestamp_ms': 1775756443519, 'update_timestamp_ms': 1775756443519, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES'